In [ ]:
# @title 1. Clone LTX-Video repo + install (native inference path)
import sys, os
if not os.path.exists('/content/ltx'):
    !git clone --depth 1 https://github.com/Lightricks/LTX-Video.git /content/ltx
sys.path.insert(0, '/content/ltx')
!pip install -q -e /content/ltx
!pip install -q imageio imageio-ffmpeg av
try:
    from ltx_video.inference import InferenceConfig
    print('OK: ltx_video importable')
except Exception as e:
    print('IMPORT FAILED:', repr(e))

In [ ]:
# @title 2. Upload your IMAGE FOLDER
# In Colab's left sidebar: New folder icon -> create 'inputs' -> upload ALL ~50 images there.
# (Drag-drop multiple files, or zip them, upload the zip, then unzip below.)
import os
IN_DIR = '/content/inputs'
os.makedirs(IN_DIR, exist_ok=True)
print('IN_DIR:', IN_DIR, '| current files:', sorted(os.listdir(IN_DIR))[:10])
# If you uploaded a zip instead, uncomment + run:
# !unzip -o /content/inputs.zip -d /content/inputs

In [ ]:
# @title 3. BATCH run LTX 2B-Distilled over ALL images (one model reload each, resumable)
import sys, os, glob
if '/content/ltx' not in sys.path:
    sys.path.insert(0, '/content/ltx')
from ltx_video.inference import infer, InferenceConfig

IN_DIR = '/content/inputs'
OUT_DIR = '/content/ltx_out'
os.makedirs(OUT_DIR, exist_ok=True)
PIPELINE_CFG = '/content/ltx/configs/ltxv-2b-0.9.6-distilled.yaml'

PROMPT = (
    'Paper-collage animation style, flat illustrated vault door made of cut paper layers, '
    'the paper pieces shift and lift slightly with a gentle hand-made stop-motion drift, '
    'soft paper shadows, warm editorial color palette, Vox-style explainer motion, '
    'subtle slow camera push-in, no 3D, no photorealism'
)
NEG = 'photorealistic, 3d render, CGI, jitter, flicker, distorted, blurry'

imgs = sorted(glob.glob(os.path.join(IN_DIR, '*.*')))
imgs = [f for f in imgs if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]
print('found', len(imgs), 'images')

done = set()
for idx, img in enumerate(imgs):
    base = os.path.splitext(os.path.basename(img))[0]
    # resumable: skip if a sidecar marker for this image already exists
    marker = os.path.join(OUT_DIR, f'.done_{base}.txt')
    if os.path.exists(marker):
        print(f'[{idx+1}/{len(imgs)}] skip (already done): {base}')
        continue
    print(f'[{idx+1}/{len(imgs)}] generating: {base}')
    cfg = InferenceConfig(
        prompt=PROMPT,
        negative_prompt=NEG,
        conditioning_media_paths=[img],
        conditioning_start_frames=[0],
        height=1280, width=704,
        num_frames=121,
        seed=42,
        pipeline_config=PIPELINE_CFG,
        output_path=OUT_DIR,
    )
    try:
        infer(config=cfg)
        open(marker, 'w').write(base)
    except Exception as e:
        print(f'  ERROR on {base}: {repr(e)}')print('BATCH DONE ->', OUT_DIR)

In [ ]:
# @title 4. Zip ALL outputs + download (one file)
import os, glob
from google.colab import files
OUT_DIR = '/content/ltx_out'
vids = sorted(glob.glob(os.path.join(OUT_DIR, 'video_output_*.mp4')))
print('total clips:', len(vids))
if vids:
    !zip -r /content/ltx_all.zip /content/ltx_out
    files.download('/content/ltx_all.zip')
    print('Download triggered: /content/ltx_all.zip')
else:
    print('No clips found -- check cell 3 output.')